In [1]:
from skimage import feature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB


In [2]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [3]:
root_folder='../data/train_data/embedding/holderried/'

In [4]:
pos_embeddings = get_embedding_birdnet(root_folder, 1)
neg_embeddings = get_embedding_birdnet(root_folder, 0)

In [5]:
df_pos = pd.DataFrame(data=pos_embeddings)
df_pos['target'] = 1

df_neg = pd.DataFrame(data=neg_embeddings)
df_neg['target'] = 0

In [6]:
df_combined = pd.concat([df_pos, df_neg], ignore_index=True, axis=0)
df_combined = df_combined.sample(frac=1, random_state=232)

In [7]:
#Generate Test and Train datasets
X = df_combined.iloc[:, :-1] #All values except the last column
y = df_combined.iloc[:, -1] #All values from the last column

#train, test, train_target, test_target = train_test_split(X, y, test_size=0.20, random_state=23)

SVM

In [13]:
model = SVC(kernel='linear', cache_size=500)

accuracies = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"Cross-validation accuracies: {accuracies}")
print(f"Mean Accuracy: {accuracies.mean():.4f}")

ap = cross_val_score(model, X, y, cv=5, scoring='average_precision')

print(f"Cross-validation AP: {ap}")
print(f"mAP: {ap.mean():.4f}")


Cross-validation accuracies: [0.99694812 0.99796541 0.99898271 0.99694812 0.99694812]
Mean Accuracy: 0.9976
Cross-validation AP: [0.99997431 0.99997843 0.9999957  0.99993959 0.99997409]
mAP: 1.0000


Random Forest

In [14]:
model = RFC(n_jobs = -1)
accuracies = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"Cross-validation accuracies: {accuracies}")
print(f"Mean Accuracy: {accuracies.mean():.4f}")

ap = cross_val_score(model, X, y, cv=5, scoring='average_precision')

print(f"Cross-validation AP: {ap}")
print(f"mAP: {ap.mean():.4f}")

Cross-validation accuracies: [0.99084435 0.99389624 0.98982706 0.98779247 0.98677518]
Mean Accuracy: 0.9898
Cross-validation AP: [0.99965937 0.99961901 0.999703   0.99962579 0.99945571]
mAP: 0.9996


XGBoost

In [15]:
model = XGB.XGBClassifier(objective='binary:logistic')

accuracies = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"Cross-validation accuracies: {accuracies}")
print(f"Mean Accuracy: {accuracies.mean():.4f}")

ap = cross_val_score(model, X, y, cv=5, scoring='average_precision')

print(f"Cross-validation AP: {ap}")
print(f"mAP: {ap.mean():.4f}")

Cross-validation accuracies: [0.99389624 0.99084435 0.98982706 0.99287894 0.98880977]
Mean Accuracy: 0.9913
Cross-validation AP: [0.99982735 0.99987507 0.99967083 0.99952287 0.99951402]
mAP: 0.9997


In [11]:
#predictions = model.predict(test)
#ap = average_precision_score(test_target, predictions)
#print("Test set average precision:", ap)

#report=classification_report(test_target, predictions, digits=4)
#print(report)

# Only for binary classification (adjust for multi-class)
#precision, recall, thr = precision_recall_curve(test_target, predictions)


In [12]:
#plt.plot(recall, precision, marker='.')
#plt.xlabel('Recall')
#plt.ylabel('Precision')
#plt.title('Precision-Recall Curve')
#plt.show()